In [93]:
from abc import ABC, abstractmethod
import random

In [94]:
class Runnable(ABC) : 
    
    @abstractmethod
    def invoke(self,input_data ) : 
        pass 

In [95]:
# Creating a fake LLM
class fake_llm(Runnable):

    def __init__(self):
        print('llm created')

        self.response_list = [
            'Delhi is the capital of India',
            'IPL is a cricket League',
            'AI stands for Artificial Intelligence'
        ]

    def invoke(self, input_data):
        return {
            'response': random.choice(self.response_list)
        }

    def predict(self, prompt):
        return self.invoke(prompt)

In [113]:
class fake_template(Runnable):

    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def invoke(self, input_dict):
        missing = [
            variable for variable in self.input_variables
            if variable not in input_dict
        ]

        if missing:
            raise ValueError(
                f"Missing template input(s): {missing}. "
                f"Expected keys: {self.input_variables}"
            )

        return self.template.format(**input_dict)

    def format(self, input_dict):
        return self.invoke(input_dict)

In [97]:
class RunnableConnector(Runnable):

    def __init__(self, runnable_list):
        self.runnable_list = runnable_list

    def invoke(self, input_data):

        for runnable in self.runnable_list:

            input_data = runnable.invoke(input_data)

        return input_data

In [114]:
class FakeStrOptParser(Runnable) : 
    
    def __init__(self):
        pass 
    
    def invoke(self, input_data): 
        return input_data['response']

In [115]:
template = fake_template(
    template="Write a {length} poem on {topic}",
    input_variables=["length", "topic"]
)

print(template.invoke({
    'length': 'short',
    'topic': 'India'
}))

Write a short poem on India


In [116]:
llm = fake_llm()

llm created


In [117]:
parser = FakeStrOptParser()

In [118]:
chain = RunnableConnector([template, llm, parser])

In [119]:
chain.invoke({'length' : 'short', 'topic' : 'India'})

'IPL is a cricket League'

In [120]:
print("Template object:", repr(template.template))
print("Input variables:", template.input_variables)
print("Input dictionary:", {'length': 'short', 'topic': 'India'})

Template object: 'Write a {length} poem on {topic}'
Input variables: ['length', 'topic']
Input dictionary: {'length': 'short', 'topic': 'India'}


## Making 2 chains and connecting them together

In [121]:
template_1 = fake_template(
    template = "Write me a joke about {topic}",
    input_variables=['topic']
)


In [122]:
template_2 = fake_template(
    template = "Explain this following joke :\n{response}",
    input_variables=['response']
)


In [123]:
llm = fake_llm()

llm created


In [129]:
# First chain: creates a prompt about the topic
chain1 = RunnableConnector([
    fake_template(
        template="Write me a joke about {topic}",
        input_variables=["topic"]
    ),
    llm,
    parser
])

# Second chain: creates a prompt using the first chain's output
chain2 = RunnableConnector([
    fake_template(
        template="Explain this following joke: {response}",
        input_variables=["response"]
    ),
    llm,
    parser
])

In [ ]:
final_chain = RunnableConnector([chain1,chain2])
final_chain.invoke({'topic' : 'politics'})